In [72]:
import openseespy.opensees as ops
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import opsvis as opsv
from vfo import vfo
from streng.codes.eurocodes.ec8.cls.seismic_action.spectra import SpectraEc8
from streng.ppp.sections.geometry.tee import TeeSectionGeometry
from streng.ppp.sections.geometry.rectangular import RectangularSectionGeometry
from streng.codes.eurocodes.ec2.raw.ch5.geometric_data import effective_width


In [73]:
# Δεδομένα      
# Μονάδες: kN, m, C

# Διαστάσεις

L = 7             # Μήκος δοκού (m)
H = 3             # Mήκος υποστυλώματος (m)
a = L/4           # Απόσταση φορτίου από ακραίο υποστύλωμα (m)
L1 = L/2          # Μήκος τμήματος δοκού μεταξύ υποστυλωμάτων (m)
cnom = 0.05       # Επικάλυψη (m)

# Δοκός - Τυπική διατομή Τ
bw = 0.3         
hf = 0.15        
hb = 0.5         

# Για μονό άνοιγμα με πλαισιακή σύνδεση (μονολιθική με υποστυλώματα)
l0 = effective_width.l0(l2=L, zero_moments_case=1)  
b1 = b2 = L/2 - bw/2  
beff1 = beff2 = effective_width.beffi(b1, l0)
_b = bw + b1 + b2
beff = effective_width.beff(bw, beff1, beff2, _b)

# Υποστύλωματα
Bc = 0.5  # Πλάτος (m)
Hc = 0.5  # Ύψος (m)

# Φορτία
hep = 0.05  # Πάχος επίστρωσης    (m)
htoix = 2.5 # Ύψος τοιχοπληρώσεων (m)

# Ειδικά βάρη 
gskyr = 25   # kN/m3
gepist = 20  # kN/m3
gtoix = 3.6  # kN/m2

# Φορτία Δοκού

# Μόνιμα φορτία πλακών

g1 = 1.5          # kN/m2
gIB = gskyr * hf  # kN/m2           
gol = g1 + gIB    # kN/m2

gd = gol * (a/2 + a/2)    # kN/m
gdtoix = gtoix * htoix    # kN/m
gdIB = gskyr * bw * (hb-hf)
g = gd + gdtoix + gdIB    # kN/m
print(f'g = {g:.3f} kN/m')

# Ωφέλιμα φορτία πλακών
Q = 5                    # kN/m2
q = Q * (a/2 + a/2)      # kN/m
print(f'q = {q:.3f} kN/m')

# Υλικά - Σκυρόδεμα C20/25 - Χάλυβας B500C

# Σκυρόδεμα 
fck = 20                            # Χαρακτηριστική Θλιπτική αντοχή (Mpa)
fcm = fck + 8                       # Μέση θλιπτική αντοχή (Mpa)
Ecm = round(22*(fcm/10)**0.3, 1)    # Μέτρο ελαστικότητας (GPa) 
U = 0.0                             # Συντελεστής Poisson
E = Ecm * 10**6                     # (Pa)
G = E / (2*(1+U))                   # Μέτρο διάτμησης (Pa)

# Διατομές

# Δοκός 
tbeam = TeeSectionGeometry(bw = bw, h = hb, beff=beff, hf = hf)
A_tbeam = tbeam.area
Iz_tbeam = tbeam.moment_of_inertia_xx * 0.5
Avy_tbeam = tbeam.shear_area_2 * 0.5

# Υποστύλωματα 
rect_col = RectangularSectionGeometry(b=Bc, h=Hc)
A_col = rect_col.area
Iz_col = rect_col.moment_of_inertia_xx * 0.5
Avy_col = rect_col.shear_area_2 * 0.5

# Μάζα
mass = (g+0.3*q)*L*2 / 9.81


g = 20.812 kN/m
q = 8.750 kN/m


In [74]:
print(f'mass = {mass:.2f} t')
print(f'H = {H:.3f} m')
print(f'L = {L:.3f} m')
print(f'E = {E} kPa')
print(f'G = {G} kPa')


mass = 33.45 t
H = 3.000 m
L = 7.000 m
E = 30000000.0 kPa
G = 15000000.0 kPa


In [75]:
# Μοντελοποίηση 
ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)

# Nodes 
ops.node(1, 0, 0)
ops.node(2, 0, H)
ops.node(3, L, 0)
ops.node(4, L, H)


# Στηρίξεις
ops.fix(1, 1, 1, 1)
ops.fix(4, 1, 1, 1)

ops.geomTransf('Linear', 1) 
  
ops.mass(int(2), mass, 1.0e-10, 1.0e-10)

ops.equalDOF(2, 3, 1)  

In [76]:
# Elements
# Υποστύλωματα
ops.element('elasticTimoshenkoBeam', 1, 1, 2, A_col, E, G, Iz_col, Avy_col, 1)
ops.element('elasticTimoshenkoBeam', 2, 3, 4, A_col, E, G, Iz_col, Avy_col, 1)
# Δοκός (ενιαία όπως αρχικά)
ops.element('elasticTimoshenkoBeam', 3, 2, 4, A_tbeam, E, G, Iz_tbeam, Avy_tbeam, 1)

elem_type = {1:'Column', 2:'Column', 3:'Beam'}


numEigen = 1
eigenValues = ops.eigen('-genBandArpack', numEigen)


In [77]:
_periods = []
for i in range(0, numEigen):
    lamb = eigenValues[i]
    period = 2 * np.pi / np.sqrt(lamb)
    _periods.append(period)
    print(f'Period {i+1} = {period:.4f}s')


Period 1 = 0.0641s
